In [12]:
import numpy as np
from numpy import random
import matplotlib.pyplot as plt
import math
import pandas as pd

%matplotlib inline

In [17]:
def propensity(k,*args):
    
    for i in args:
        k = k*i
    return k


def ribosomebinding(a,b,c):
    
    if a>0 and b>0:
        a-=1
        b-=1
        c+=1
    return a,b,c


def releaserbs(a,b,c):
    
    if a>0:
        a-=1
        b+=1
        c+=1
    return a,b,c


def readthrough(a,b):
    
    if a>0:
        a-=1
        b+=1
    return a,b


def slowcodon_index(codon_no, slowcodon_no):
    
    position_list=list(range(codon_no))
    random.seed()
    primary_list=random.choice(position_list, slowcodon_no, replace=False)
    index_list=sorted(primary_list)
    
    codon_str=''
    for i in position_list:
        if i in index_list:
            codon_str+='1'
        else:
            codon_str+='0'
    
    return index_list,codon_str


def binary2decimal(num_str):
    
    deci_no=int(num_str, 2)
    
    return deci_no


def decimal2binary(deci_num,codon_no):
    
    bin_no=np.binary_repr(deci_num)
    
    for i in range(codon_no-len(bin_no)):
        bin_no='0'+bin_no
    
    return bin_no


def get_reconstant(bin_no):
     
    bin_list=list(bin_no)
#     print('BIN LIST', bin_list,len(bin_list))
    
    reconstant_list=[0.002]
    
    for i in bin_list:
        if i=='0':
            reconstant_list.append(12)
        elif i=='1':
            reconstant_list.append(2)
        else:
            print('ERROR')
    
    return reconstant_list


def directmethod(totalcodon_no, complex_list, reconstant_list, total_rib, rna, t1, occupied_pos):
    
    n=totalcodon_no
    
    rib=total_rib
    rna_rbs=rna
    
    prop_list=[] 
    prop1=propensity(reconstant_list[0],rib,rna_rbs)

    prop_list.append(prop1)
     
    for i in range(1,n):   # calculate the propensity of ribosome reading through the mRNA
        prop_list.append(propensity(reconstant_list[i],complex_list[i-1]))
        
    if occupied_pos!=[]:  # occupied_pos stores the occupied codon positions, exclude them by resetting their propensities to 0
        new_prop = []   # will be used to store the propensities of reactions occurred in available codon positions
        for i, p in enumerate(prop_list):   # get the indexes and values of elements in prop_list
            if i not in occupied_pos:  # if the index of element is not in occupied_pos, it means that the position is available
                new_prop.append(p)  # just store the value into new_prop list
            else:   # else means the index of element is in occupied_pos, it means that the position has been occupied
                new_prop.append(0)  # just append 0 into the new_prop (means the reaction cannnot occur)
    else:
        new_prop = prop_list
        
    sumprop=sum(new_prop)
    prob_list=[]
    
    if sumprop > 0:
        for i in new_prop: 
            prob_list.append(i/sumprop)  # calculate the probablity of each reaction
    
    if sum(prob_list)!=0:
        u=random.choice(np.arange(0,n),p=prob_list)  # choose u 
        tau=random.exponential(1/sumprop)    # choose time tau
        
        if u==0: # u==0, means ribosome binding into mRNA, update the number of ribosome, RNA_RBS and Ribosome-RNA_RBS complex
            rib,rna_rbs,complex_list[0]=ribosomebinding(rib,rna_rbs,complex_list[0]) 
            
        elif u==1: # u==1, means ribosome read through the second codon and release the RBS
            complex_list[0],complex_list[1],rna_rbs=releaserbs(complex_list[0],complex_list[1],rna_rbs) 
            
        elif u==n-1: # u==n-1, means the ribosome encounters the STOP or the last codon
            complex_list[u-1],rib=readthrough(complex_list[u-1],rib) 
            
        else:# else means the ribosome reads through other codon positions, regarded as the transition 
            # of Ribosome-RNA complex[i]-->Ribosome-RNA complex[i+1]
            complex_list[u-1],complex_list[u]=readthrough(complex_list[u-1],complex_list[u])
        
    newcomplex_list=[]  
    for i in range(len(complex_list)): # get and store the updated number of Ribosome-RNA complex[i] into newcomplex_list
        newcomplex_list.append(complex_list[i])
        
    rib1=rib  # get the updated number of free ribosome
    rna_rbs1=rna_rbs # get the updated number of RNA_RBS
    
    
    data_dict={'newcomplex_list':pd.Series([newcomplex_list]), 'rib1':pd.Series([rib1]), \
             'rna_rbs1':pd.Series([rna_rbs1]), 'tau':pd.Series([tau]), 'u': pd.Series([u])}
    
    data=pd.DataFrame.from_dict(data_dict, orient='columns')
    
#     return newcomplex_list, rib1, rna_rbs1, tau, u
    return data


def simulation(reconstant_list, total_rib,t1):
    
    t0=t1  # use t0 to store the initial value of t1 (stimulation starting time)
    t1_list=[]  # will be used to store the updating reaction time t1
    
#     n=codon_no+1

    n=len(reconstant_list)
    
#     reconstant_list=get_reconstant(codon_no, slowcodon_no)[0]
    
    complex_list=[0]*(n-1)  # set the initial value of complex_list (all the elements are 0)
    
    rib=total_rib # get the initial number of free ribosome
    
    rna_rbs=1  # the initial number of mRNA-RBS is 1 as only 1 mRNA is involved in the simulation
    
    result={}  # will be used to store the time and positions of each individual reaction
    
    ids=0  # will be used as the ID of new reaction (also as the name of reaction)
    u_list=[]   # will be used to store the generated u at each step
    
    occupied_pos=[]  # will be used to store the codon postions that has been occupied (reaction cannot occur)

    number=0   # the number of proteins that have been translated completely
    
    stall_count=0
    
    
    while number<100:  

        y=directmethod(n, complex_list, reconstant_list, rib, rna_rbs, t1, occupied_pos)  # run direct method
        u=list(y.u)[0]
        
#         print('U', u)
    
        if u_list==[]:    # at first u_list is empty, 
            complex_list=list(y.newcomplex_list)[0]
            rib=list(y.rib1)[0]
            rna_rbs=list(y.rna_rbs1)[0]
            
            t1+=list(y.tau)[0]
            t1_list.append(t1)  # store the updated time 
            
            u_list.append(u)  # store the u generated from this step (then u_list is not empty)
            
            ids+=1  # start a new reaction (ids will be used as the name of new reaction)
            result[ids]=[(t1,u)]  # store the tuple of (reaction time, u) into dictionary (the ids will be the key)
            
            occupied_pos=[u]  # update the occupied_pos
            
        else:  # then u_list is not empty 
            complex_list=list(y.newcomplex_list)[0]
            rib=list(y.rib1)[0]
            rna_rbs=list(y.rna_rbs1)[0]

            u_list.append(u)
            
            if u==0:   # if u==0, means a new peptide translation reaction has occurred
                ids+=1   # use the new ids as the name of this new reaction
#                 t1 += y[3]  # update the reaction time
                t1+=list(y.tau)[0]
                t1_list.append(t1)  # store the updated reaction time t1 into t1_list
                result[ids]=[(t1,u)]  # store the tuple of (reaction time, u) into dictionary (the new ids will be the key)
                    
            else:  # if u!=0, means one of the current peptide reaction is continuing
#                 t1 += y[3]  # updating the reaction time
                t1+=list(y.tau)[0]
                t1_list.append(t1)  # store the updated reaction time t1 into t1_list
                for k, v in result.items(): # then check which reaction is continuing,
                    if v[-1][-1]==u-1: # v[-1][-1] is the last codon position of each individual reaction, if v[-1][-1]==u-1,
                        result[k].append((t1, u)) # it means this peptide translation reaction is occuring, just append the 
                                                 # tuple of (reaction time, u) into result dictionary under this key, and also
                        break                   # append the translated amino acid into aa_result dictionary under this key
                        
            occupied_pos = [v[-1][-1] for v in result.values() if v[-1][-1]!=n-1]
            occupied_pos = [v - 9 for v in occupied_pos if v>8]+[v - 8 for v in occupied_pos if v>7]+\
            [v - 7 for v in occupied_pos if v>6]+[v - 6 for v in occupied_pos if v>5]+[v - 5 for v in occupied_pos if v>4]+\
            [v - 4 for v in occupied_pos if v>3]+[v - 3 for v in occupied_pos if v>2]+[v - 2 for v in occupied_pos if v>1]+\
            [v - 1 for v in occupied_pos if v>0] + occupied_pos  
            
#             print('OCC', occupied_pos)
            
            if u==n-1:  # if u==n-1, it means the one protein has been translated completely
                number+=1  # update the number of proteins that have been translated completely.
            else:
                pass
            
            if u+1 in occupied_pos:
                stall_count+=1   
                
    data_dict={'result':pd.Series([result]), 'u_list':pd.Series([u_list]), \
             't1':pd.Series([t1]), 'stall_count':pd.Series([stall_count])}
    
    data=pd.DataFrame.from_dict(data_dict, orient='columns')    
   
    return data



# def calculate_oneprotein(codon_no,slowcodon_no, reconstant_list, total_rib, exclude_pr, t1):

def calculate_oneprotein(reconstant_list, total_rib, exclude_pr, t1):    
    
    y=simulation(reconstant_list, total_rib, t1)
    
    stall_count=list(y.stall_count)[0]
    
    protein_dic={}

#     for k, v in y[0].items():

    for k, v in list(y.result)[0].items():
        if v[-1][-1]==len(reconstant_list)-1:
            protein_dic[k]=v[-1][0]-v[0][0]
         
    keylist=list(protein_dic.keys())
    keylist.sort()
    
    time_list=[]
    
    for k in keylist:
        time_list.append(protein_dic[k])
    
#     print('TIME', time_list, len(time_list))
    
    stable_list=time_list[exclude_pr:]
#     print('STABLE', stable_list, len(stable_list))
    
    sum_stable=sum(stable_list)
#     print('SUM', sum_stable)
    
    data_dict={'protein_dic':pd.Series([protein_dic]), 'time_list':pd.Series([time_list]), \
             'stable_list':pd.Series([stable_list]), 'sum_stable':pd.Series([sum_stable]),\
               'stall_count': pd.Series([stall_count])}
    
    data=pd.DataFrame.from_dict(data_dict, orient='columns')    
    
    return data
                
    
    
def repeat_calculation(reconstant_list, total_rib,exclude_pr, t1, repeat_time):
    
    total_time_list=[]
    total_sum=[]
    
    stall_list=[]
    
    
    for i in range(repeat_time):
        y=calculate_oneprotein(reconstant_list, total_rib, exclude_pr, t1)
        total_time_list.append(list(y.stable_list)[0])
        total_sum.append(list(y.sum_stable)[0])
        stall_list.append(list(y.stall_count)[0])
    
#     print('TOTAL', total_time_list, len(total_time_list))
#     print('TOTAL SUM', total_sum, len(total_sum))
    
    pro_num=codon_no-exclude_pr
    
    s_list=[]
    for j in range(pro_num):
#         s_list=[]
        for i in total_time_list:
#             print(i[j])
            s_list.append(i[j])
     
#     print('S_LIST', s_list, len(s_list))
            
    mean_time=sum(s_list)/(pro_num*repeat_time)
    std_time=np.std(s_list, ddof=1)
#     se_time=np.std(s_list,ddof=1)/math.sqrt(80*repeat_time)
    se_time=std_time/math.sqrt(pro_num*repeat_time)
    
    mean_stall=sum(stall_list)/(repeat_time)
    std_stall=np.std(stall_list, ddof=1)
    se_stall=std_stall/math.sqrt(repeat_time)
            
#     print('MEAN', meantime_list,len(meantime_list))
#     print('STD',std_list,len(std_list))
#     print('SE',se_list,len(se_list))
    
#     print('STALL_LIST', stall_list, len(stall_list))

    data_dict={'mean_time':pd.Series([mean_time]), 'std_time':pd.Series([std_time]), \
             'se_time':pd.Series([se_time]), 'mean_stall':pd.Series([mean_stall]),\
               'std_stall': pd.Series([std_stall]), 'se_stall': pd.Series([se_stall])}
    
    data=pd.DataFrame.from_dict(data_dict, orient='columns')    
    
#     return mean_time,std_time,se_time, mean_stall, std_stall, se_stall
    return data
        

In [18]:
codon_no=100
slowcodon_no=20
exclude_pr=20
rib=50
t1=0
repeat_time=3

total_deci_no=[]
for i in range(3):
    y=slowcodon_index(codon_no, slowcodon_no)
    deci_no=binary2decimal(y[1])
    total_deci_no.append(deci_no)
    
print('TOTAL DECI NO', total_deci_no)

total_reconstant_list=[]
for j in total_deci_no:
    z=decimal2binary(j,codon_no)
    reconstant_list=get_reconstant(z)
    print(len(reconstant_list))
    total_reconstant_list.append(reconstant_list)

print('TOTAL RE', total_reconstant_list)


total_mean_time=[]
total_se_time=[]
total_mean_stall=[]
total_se_stall=[]

for i in total_reconstant_list:
    print('No,',total_reconstant_list.index(i))
    z=repeat_calculation(i, rib, exclude_pr, t1, repeat_time)
    total_mean_time.append(list(z.mean_time)[0])
    total_se_time.append(list(z.se_time)[0])
    total_mean_stall.append(list(z.mean_stall)[0])
    total_se_stall.append(list(z.se_stall)[0])
    
    print('TOTAL M TIME',total_mean_time, len(total_mean_time))
    print('TOTAL SE TIME', total_se_time, len(total_se_time))
    print('TOTAL M STALL', total_mean_stall, len(total_mean_stall))
    print('TOTAL SE STALL', total_se_stall, len(total_se_stall))


TOTAL DECI NO [653991506385394448736737822720, 87903435502098065617001136129, 218380360207390562035473326097]
101
101
101
TOTAL RE [[0.002, 2, 12, 12, 12, 12, 2, 12, 12, 12, 12, 12, 2, 12, 12, 2, 12, 2, 12, 12, 2, 12, 12, 12, 2, 2, 12, 12, 12, 12, 2, 12, 12, 2, 12, 12, 12, 2, 12, 12, 12, 12, 12, 12, 2, 12, 2, 2, 2, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 2, 2, 12, 12, 2, 12, 12, 12, 12, 12, 12, 12, 2, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 2, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12], [0.002, 12, 12, 12, 2, 12, 12, 12, 2, 2, 2, 12, 12, 12, 12, 12, 12, 2, 12, 12, 12, 12, 12, 12, 12, 12, 2, 12, 12, 2, 12, 12, 12, 12, 12, 2, 2, 12, 12, 12, 12, 12, 2, 12, 12, 12, 12, 12, 12, 12, 2, 12, 2, 12, 2, 2, 12, 12, 12, 12, 12, 2, 12, 12, 2, 12, 12, 2, 12, 12, 12, 12, 12, 12, 12, 12, 2, 12, 12, 12, 12, 12, 12, 12, 12, 12, 2, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 2], [0.002, 12, 12, 2, 12, 2, 2, 12, 12, 12, 12, 12, 2, 2, 12, 2, 12, 12, 12, 12, 12, 12, 12, 